# Notebook 02 — Leakage-Controlled Preprocessing and Data Partitioning

**Project:** A Multi-Modal Deep Learning Approach for Eggshell Micro-Crack Morphology Analysis and Root-Cause Diagnosis in Poultry Breeder Farms
**Author:** M.P.N. Sharindi (Coventry Index 16110429)
**Stage:** 02 of 12 — runs after 01b, before 03

---

### What notebook 01b established

A gradient-boosted tree reached **macro F1 0.612 on the four-class task and 0.816 on binary crack
detection using capture statistics alone, with no image content whatsoever.** Against a 0.164
majority baseline that is severe. Left untreated, a convolutional network would be rewarded for
reading the photograph rather than the egg.

Further analysis of the master table identified the mechanism, and it is mostly fixable:

| Leak channel | Evidence | Removable by |
|---|---|---|
| **Black letterbox padding** | fraction of near-black pixels: Intact 0.032, Hairline 0.134, Star 0.198; padding epsilon squared = 0.140 | **tight crop to the egg** — removes it completely |
| **Global brightness, contrast, colour** | mean luminance 168 vs 143–154; red/blue ratio 4.9 vs 6.3–7.3; 100 % of the luminance effect survives crack exclusion, so it is acquisition not signal | **per-image per-channel standardisation** |
| **Absolute resolution** | min_side correlates with capture order at rho = +0.73; Intact mean 1855 px vs Star 2472 px | **fixed-size resize** (every model does this) |
| **Framing** | egg area / frame and egg bounding-box fill both rank in the top four leaking features | **tight crop** makes these near-constant |
| Residual exposure clipping and texture | survives all of the above | not removable — handled by augmentation and by the control experiments in notebook 05 |

Capture order alone predicts crack status at macro F1 0.672, and the leak only partly disappears
when whole capture sessions are held out (0.819 random versus 0.663 with 130-egg blocks). So the
confound is both a session effect **and** a within-session effect. Both are addressed below.

### What this notebook does

1. Reconstructs the true partial-egg flags from the `review_log.csv` crop boxes
2. Builds the leakage-controlled preprocessing pipeline and caches every image once
3. **Re-measures the leak on the actually preprocessed images**, not on a simulation
4. Produces the sealed test set and the cross-validation folds, under two split regimes
5. Produces a capture-matched sensitivity subset for Chapter 5

### A note on framing for the dissertation

This is not a failure to hide. Almost no published work in agricultural computer vision screens for
acquisition shortcuts at all, which is one reason reported accuracies in this field are hard to
reproduce. Finding this, quantifying it, removing what can be removed and reporting the residual
honestly makes Chapter 5 substantially stronger than a clean-looking but naive evaluation would
have been. Sections 5.2.3 and 5.10.1 are built directly from this notebook.

---
## 0. Configuration

In [ ]:
import os
os.environ["EGG_ARTEFACT_ROOT"] = r"E:\Degree\Final Research\Egg_Crack_Dataset\Eggshell-Diagnostic-Research\Artefact"

CACHE_SIDE   = 448     # cached at 448 so training can use 320/384/448 without regenerating
CROP_MARGIN  = 0.04    # margin around the egg bounding box, as a fraction of the egg's longer side
N_JOBS       = -1
TEST_FRACTION = 0.15
N_FOLDS      = 5
N_CAPTURE_BLOCKS = 10  # capture-order blocks used to balance and to build the grouped split
REBUILD_CACHE = True   # set False to reuse an existing cache

In [ ]:
import sys, json, math, warnings
from pathlib import Path
from datetime import datetime

_here = Path.cwd()
_src = next((p / "src" for p in [_here, *_here.parents] if (p / "src" / "common").is_dir()), None)
if _src is None:
    raise RuntimeError("Run from Artefact/notebooks/ or set EGG_ARTEFACT_ROOT.")
sys.path.insert(0, str(_src))

import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
from scipy import stats
from joblib import Parallel, delayed
from common import project_config as CFG

warnings.filterwarnings("ignore", category=FutureWarning)
CFG.seed_everything()

STAGE = "02_split_and_preprocessing"
OUT = CFG.results_dir(STAGE); FIG_DIR, TAB_DIR = OUT / "figures", OUT / "tables"
PROC = CFG.DATA_DIR / "processed"; PROC.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 10,
                     "axes.titlesize": 11, "axes.titleweight": "bold",
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "legend.frameon": False,
                     "figure.facecolor": "white", "savefig.facecolor": "white"})
_N = {"i": 0}; FIGS, TABS = [], []
def savefig(fig, name, cap=""):
    _N["i"] += 1; f = f"{_N['i']:02d}_{name}.png"
    fig.savefig(FIG_DIR / f, bbox_inches="tight"); FIGS.append({"file": f, "caption": cap})
    print(f"  [figure] {f}")
def savetable(df, name, cap=""):
    p = TAB_DIR / f"{name}.csv"; df.to_csv(p); TABS.append({"file": p.name, "caption": cap})
    print(f"  [table ] {p.name}  {df.shape}"); return df
S = {"stage": STAGE, "generated_at": datetime.now().isoformat(timespec="seconds")}

master = pd.read_csv(PROC / "master_table.csv")
def resolve_paths(df):
    if "path" in df.columns and df["path"].notna().all() and Path(str(df["path"].iloc[0])).is_file():
        return df
    for cand in ("path", "file_name", "filename", "file"):
        if cand in df.columns:
            p = df[cand].astype(str).map(lambda v: Path(v) if Path(v).is_absolute()
                                         else CFG.IMAGES_DIR / Path(v).name)
            if p.map(lambda q: q.is_file()).all():
                df["path"] = p.astype(str); return df
    df["path"] = df["egg_id"].map(lambda e: str(CFG.IMAGES_DIR / f"{e}.jpg"))
    miss = int(sum(not Path(q).is_file() for q in df["path"]))
    if miss: raise FileNotFoundError(f"{miss} images missing under {CFG.IMAGES_DIR}")
    return df
master = resolve_paths(master)
master["capture_order"] = master["egg_id"].str.extract(r"E(\d{3})").astype(int)
master["label4"] = pd.Categorical(master["label4"], categories=CFG.LABEL4, ordered=True)
master["age_class"] = pd.Categorical(master["age_class"], categories=CFG.AGE_ORDER, ordered=True)
print(f"{len(master)} eggs loaded")

---
## 1. True partial-egg flags from the crop boxes

`review_log.csv` records the crop rectangle chosen for each reviewed image as `box_x, box_y, box_w,
box_h` in the coordinate space of the original 3672 x 8160 photograph, plus a `status` of `OK` or
`OK_MANUAL`. There is no `PARTIAL` value, which is why notebook 01b found none — the information is
there, but encoded in the geometry rather than the label.

A crop box that is clamped against an edge of the original frame means the egg extended beyond the
photograph and the saved image is a partial egg. A box whose width and height differ means the
square output was produced by padding. Both are recovered here.

In [ ]:
ORIG_W, ORIG_H = 3672, 8160     # original capture resolution, portrait

review = None
if CFG.REVIEW_LOG_CSV.is_file():
    review = pd.read_csv(CFG.REVIEW_LOG_CSV, encoding="utf-8-sig")
    print(f"review_log.csv: {review.shape[0]} rows, columns = {list(review.columns)}")
else:
    print(f"review_log.csv not found at {CFG.REVIEW_LOG_CSV} -- falling back to the Otsu detector")

if review is not None:
    review["egg_id"] = review.iloc[:, 0].astype(str).str.extract(r"(E\d{3})", expand=False)
    review = review.dropna(subset=["egg_id"]).drop_duplicates("egg_id").set_index("egg_id")
    need = {"box_x", "box_y", "box_w", "box_h"}
    if not need <= set(review.columns):
        raise KeyError(f"review_log.csv is missing {need - set(review.columns)}")

    EDGE = 2
    r = review
    clamped_left   = r["box_x"] <= EDGE
    clamped_top    = r["box_y"] <= EDGE
    clamped_right  = (r["box_x"] + r["box_w"]) >= ORIG_W - EDGE
    clamped_bottom = (r["box_y"] + r["box_h"]) >= ORIG_H - EDGE
    r["n_edges_clamped"] = (clamped_left.astype(int) + clamped_top.astype(int)
                            + clamped_right.astype(int) + clamped_bottom.astype(int))
    r["box_partial"] = r["n_edges_clamped"] > 0
    r["box_aspect"] = r["box_w"] / r["box_h"]
    r["box_padded"] = (r["box_aspect"] - 1).abs() > 0.02

    for c in ["box_x", "box_y", "box_w", "box_h", "box_aspect", "box_partial",
              "box_padded", "n_edges_clamped"]:
        master[c] = master["egg_id"].map(r[c])
    if "status" in r.columns:
        master["review_status"] = master["egg_id"].map(r["status"])

    master["box_partial"] = master["box_partial"].fillna(False).astype(bool)
    master["box_padded"] = master["box_padded"].fillna(False).astype(bool)

    tab = (master.groupby("label4", observed=True)
           .agg(n=("egg_id", "size"),
                in_review_log=("box_partial", lambda s: int(master.loc[s.index, "box_w"].notna().sum())),
                partial_from_box=("box_partial", "sum"),
                padded_from_box=("box_padded", "sum"),
                otsu_touches_border=("egg_touches_border", "sum")))
    savetable(tab, "t01_partial_from_crop_boxes",
              "Partial and padded eggs recovered from the review_log crop geometry")
    print(tab.to_string())
    S["partial_from_box"] = tab.reset_index().astype(str).to_dict("records")
    print(f"\ntotal partial eggs recovered from crop geometry: {int(master['box_partial'].sum())}")
    print(f"the Otsu detector in notebook 01 found {int(master['egg_touches_border'].sum())}")
    print("The crop geometry is authoritative; the Otsu count was an undercount because an egg can")
    print("be cut off by the original frame without its segmented region touching the cropped edge.")
else:
    master["box_partial"] = master["egg_touches_border"].astype(bool)
    master["box_padded"] = ~master["is_square"].astype(bool)

---
## 2. The leakage-controlled preprocessing pipeline

Four steps, each closing a leak channel identified in notebook 01b.

1. **Segment the egg** with the same Otsu procedure used throughout, and take its bounding box.
2. **Expand that box to a square** centred on the egg, side = longer egg dimension x (1 + margin).
   Where the square runs past the image edge, pad by **edge replication, never with black**. This is
   the critical change: black bars currently occupy a median 17 % of Star images and 0 % of Intact
   images, and `frac_clipped_lo` — the fraction of near-black pixels — is one of the strongest
   leaking features in the whole dataset (Intact 0.032, Star 0.198).
3. **Resize to 448 x 448** isotropically. Aspect is preserved by construction in step 2, so crack
   elongation — the single most discriminative geometric feature, epsilon squared = 0.51 — is not
   distorted. Absolute resolution information is destroyed.
4. **Per-image per-channel standardisation** is applied at training time, not baked into the cache,
   so that notebook 03 can ablate it. Caching raw uint8 keeps the cache reusable.

The cache is a single memory-mapped array: 652 x 448 x 448 x 3 uint8 = **392 MB**, which sits
comfortably in 16 GB of RAM. Decoding a 1700–4400 px JPEG costs 50–150 ms; reading from the cache
costs under a millisecond. Across 30 epochs and 5 folds this is the difference between hours and
minutes on the RTX 3050.

In [ ]:
MASK_SIDE = 512

def egg_mask_and_bbox(gray):
    """Otsu segmentation of the candled egg: full-resolution boolean mask plus bounding box."""
    H, W = gray.shape
    small = cv2.GaussianBlur(cv2.resize(gray, (MASK_SIDE, MASK_SIDE), interpolation=cv2.INTER_AREA),
                             (5, 5), 0)
    _, m = cv2.threshold(small, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((7, 7), np.uint8))
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8))
    n, lab, st, _ = cv2.connectedComponentsWithStats(m, 8)
    if n <= 1:
        return np.ones((H, W), bool), (0, 0, W, H)
    k = 1 + int(np.argmax(st[1:, cv2.CC_STAT_AREA]))
    msmall = (lab == k).astype(np.uint8)
    x, y, bw, bh = st[k, :4]
    sx, sy = W / MASK_SIDE, H / MASK_SIDE
    mask = cv2.resize(msmall, (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)
    return mask, (int(x * sx), int(y * sy), int(bw * sx), int(bh * sy))


def neutral_fill(img, mask, seed):
    """Replace everything outside the egg with the egg's own colour statistics.

    This is the fix for the leak that survived the first version of this notebook. Expanding the
    crop to a square pulled in the black letterbox bars that crop_eggs.py had baked INTO the source
    image, so `frac_clipped_lo` -- the fraction of near-black pixels, one of the strongest leaking
    features -- was unchanged. Filling with the per-image egg median plus matched noise removes
    every near-black pixel while leaving no suspiciously flat region for the network to key on.
    """
    out = img.copy()
    egg = img[mask]
    if egg.size == 0:
        return out
    med = np.median(egg, axis=0)
    sd = np.clip(egg.std(axis=0), 1.0, None)
    bg = (~mask)
    n_bg = int(bg.sum())
    if n_bg:
        rng = np.random.default_rng(seed)
        noise = rng.normal(0.0, sd * 0.5, size=(n_bg, 3))
        out[bg] = np.clip(med + noise, 0, 255).astype(np.uint8)
    return out


def interior_square(mask):
    """Largest axis-aligned square that fits entirely inside the egg.

    The Chebyshev distance transform gives, at every pixel, the half-side of the largest square
    centred there that stays inside the mask. Its maximum is the answer.
    """
    dt = cv2.distanceTransform(mask.astype(np.uint8), cv2.DIST_C, 3)
    cy, cx = np.unravel_index(int(np.argmax(dt)), dt.shape)
    half = int(dt[cy, cx])
    return cx, cy, max(half, 8)


def crack_mask_from_polygons(crack_region, shape):
    h, w = shape
    m = np.zeros((h, w), np.uint8)
    if isinstance(crack_region, str) and crack_region.strip():
        for o in json.loads(crack_region):
            p = o.get("points", [])
            if len(p) >= 3:
                pts = np.round(np.asarray(p, float) / 100.0 * np.array([w, h])).astype(np.int32)
                cv2.fillPoly(m, [pts], 1)
    return m


def preprocess(path, crack_region, row_seed, side=CACHE_SIDE, margin=CROP_MARGIN,
               variant="neutral_fill"):
    """Return the preprocessed image plus provenance metadata.

    variant 'neutral_fill' : square crop centred on the egg, background replaced by egg statistics.
                             Keeps the whole egg, so no annotated crack is ever lost.
    variant 'interior'     : largest square lying entirely inside the egg. Removes the silhouette
                             completely, at the cost of clipping cracks near the egg's edge.
    """
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        return None, {"ok": False}
    H, W = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mask, (x, y, bw, bh) = egg_mask_and_bbox(gray)

    cm = crack_mask_from_polygons(crack_region, (H, W))
    crack_total = float(cm.sum())

    if variant == "interior":
        cx, cy, half = interior_square(mask)
        x0, y0, x1, y1 = cx - half, cy - half, cx + half, cy + half
    else:
        img = neutral_fill(img, mask, row_seed)
        cx, cy = x + bw / 2.0, y + bh / 2.0
        half = max(bw, bh) * (1.0 + margin) / 2.0
        x0, y0 = int(round(cx - half)), int(round(cy - half))
        x1, y1 = int(round(cx + half)), int(round(cy + half))

    pl, pt = max(0, -x0), max(0, -y0)
    pr, pb = max(0, x1 - W), max(0, y1 - H)
    crop = img[max(y0, 0):min(y1, H), max(x0, 0):min(x1, W)]
    if crop.size == 0:
        return None, {"ok": False}
    if pl or pt or pr or pb:
        crop = cv2.copyMakeBorder(crop, pt, pb, pl, pr, cv2.BORDER_REPLICATE)

    kept = float(cm[max(y0, 0):min(y1, H), max(x0, 0):min(x1, W)].sum())
    out = cv2.resize(crop, (side, side), interpolation=cv2.INTER_AREA)
    g = cv2.cvtColor(out, cv2.COLOR_BGR2GRAY)
    meta = {"ok": True, "pre_bbox_w": bw, "pre_bbox_h": bh,
            "pre_dark_frac": float((g <= 8).mean()),
            "pre_crack_retained": (kept / crack_total) if crack_total > 0 else np.nan,
            "pre_ran_off_edge": bool(pl or pt or pr or pb)}
    return cv2.cvtColor(out, cv2.COLOR_BGR2RGB), meta

### Choosing the crop variant

Two variants are built and compared, because the right trade-off is an empirical question.

* **neutral_fill** keeps the entire egg and replaces everything outside it with the egg's own colour
  statistics. No annotated crack is ever lost. The egg silhouette remains visible.
* **interior** takes the largest square lying entirely inside the egg, so the silhouette, the
  background and any truncation are gone altogether. The cost is that cracks near the egg's edge get
  clipped.

Crack retention is measured per egg by rasterising the annotation polygons and counting what
survives the crop, so the cost of the aggressive variant is a number rather than a guess.

In [ ]:
def build_cache(variant, side=CACHE_SIDE):
    path = PROC / f"cache_{variant}_{side}.npy"
    arr = np.lib.format.open_memmap(path, mode="w+", dtype=np.uint8,
                                    shape=(len(master), side, side, 3))
    res = Parallel(n_jobs=N_JOBS, verbose=1)(
        delayed(preprocess)(r.path, r.crack_region, i, side, CROP_MARGIN, variant)
        for i, r in enumerate(master.itertuples()))
    metas = []
    for i, (im, meta) in enumerate(res):
        if im is not None:
            arr[i] = im
        metas.append(meta)
    arr.flush(); del arr
    return path, pd.DataFrame(metas)

CACHES, METAS = {}, {}
for variant in ["neutral_fill", "interior"]:
    if REBUILD_CACHE or not (PROC / f"cache_{variant}_{CACHE_SIDE}.npy").is_file():
        print(f"\nbuilding '{variant}' cache ...")
        p, m = build_cache(variant)
    else:
        p = PROC / f"cache_{variant}_{CACHE_SIDE}.npy"
        m = pd.DataFrame(index=range(len(master)))
        print(f"reusing {p.name}")
    CACHES[variant] = np.load(p, mmap_mode="r")
    METAS[variant] = m
    if "pre_dark_frac" in m:
        print(f"  near-black pixel fraction after '{variant}': "
              f"median {m['pre_dark_frac'].median():.4f}, max {m['pre_dark_frac'].max():.4f}")

ret = METAS["interior"].get("pre_crack_retained")
if ret is not None:
    rr = pd.concat([master[["label4"]].reset_index(drop=True), ret], axis=1).dropna()
    rtab = rr.groupby("label4", observed=True)["pre_crack_retained"].agg(
        n="size", median="median", mean="mean",
        pct_fully_kept=lambda s: round(100 * (s >= 0.99).mean(), 1),
        pct_mostly_kept=lambda s: round(100 * (s >= 0.80).mean(), 1)).round(3)
    savetable(rtab, "t05_crack_retention_interior_crop",
              "Annotated crack area retained by the interior-square crop variant")
    S["crack_retention_interior"] = rtab.reset_index().astype(str).to_dict("records")
    print("\ncrack area retained by the 'interior' variant:")
    print(rtab.to_string())

# near-black pixels by class, the feature that leaked hardest
dark = pd.DataFrame({"label4": master["label4"].values})
for v in CACHES:
    if "pre_dark_frac" in METAS[v]:
        dark[v] = METAS[v]["pre_dark_frac"].values
if "frac_clipped_lo" in master:
    dark["before"] = master["frac_clipped_lo"].values
dtab = dark.groupby("label4", observed=True).median().round(4)
savetable(dtab, "t06_dark_fraction_by_variant",
          "Fraction of near-black pixels by class, before and after each crop variant")
S["dark_fraction_by_variant"] = dtab.reset_index().astype(str).to_dict("records")
print("\nmedian near-black pixel fraction (the strongest single leak channel):")
print(dtab.to_string())

In [ ]:
CACHE_PATH = PROC / f"cache_neutral_fill_{CACHE_SIDE}.npy"
INDEX_PATH = PROC / f"cache_{CACHE_SIDE}_index.csv"

# the primary cache is the neutral-fill variant; 'interior' is kept for the ablation
cache = CACHES["neutral_fill"]
for v, m in METAS.items():
    if "ok" in m:
        master = pd.concat([master.reset_index(drop=True),
                            m.add_prefix(f"{v}_")], axis=1)
master[["egg_id", "label4", "age_class", "capture_order"]] \
    .assign(cache_row=range(len(master))).to_csv(INDEX_PATH, index=False)
print(f"primary cache: {CACHE_PATH.name}, shape {cache.shape}, dtype {cache.dtype}")
print(f"ablation cache: cache_interior_{CACHE_SIDE}.npy")

In [ ]:
# --- visual confirmation: before and after, for one egg per class ---------
fig, axes = plt.subplots(3, 4, figsize=(14, 10.6))
for j, cls in enumerate(CFG.LABEL4):
    sub = master[master["label4"].eq(cls)]
    key = "frac_clipped_lo" if "frac_clipped_lo" in sub else None
    pick = sub.loc[sub[key].idxmax()] if key else sub.iloc[0]
    i = int(pick.name)
    raw = cv2.cvtColor(cv2.imread(str(pick["path"])), cv2.COLOR_BGR2RGB)
    panels = [(raw, "before"),
              (CACHES["neutral_fill"][i], "neutral fill"),
              (CACHES["interior"][i], "interior square")]
    for r, (im, lab) in enumerate(panels):
        axes[r, j].imshow(im)
        axes[r, j].set_title(f"{pick['egg_id']} — {cls}\n{lab}" if r == 0 else lab, fontsize=9)
        axes[r, j].set_xticks([]); axes[r, j].set_yticks([]); axes[r, j].grid(False)
fig.suptitle("Removing the black letterbox padding that leaked class", y=1.0, fontsize=12)
fig.tight_layout()
savefig(fig, "preprocessing_before_after",
        "Worst-padded example per class under each crop variant")
plt.show()

---
## 3. Re-measuring the leak on the actually preprocessed images

Notebook 01b simulated each preprocessing policy by dropping the features it would destroy. That is
an estimate. Here the capture statistics are **recomputed from the real cached images**, and the
probe is run again. This number is what goes in the dissertation.

Statistics are recomputed twice: on the cached images as stored, and on the cached images after
per-image per-channel standardisation, which is the form the network will actually see.

In [ ]:
def post_stats(i, standardise, which="neutral_fill"):
    src = CACHES[which]
    im = np.asarray(src[i], dtype=np.float32)
    if standardise:
        for c in range(3):
            ch = im[:, :, c]
            im[:, :, c] = (ch - ch.mean()) / (ch.std() + 1e-6)
    g = im.mean(axis=2)
    lap = cv2.Laplacian(g.astype(np.float32), cv2.CV_32F)
    raw = np.asarray(src[i])
    g8 = cv2.cvtColor(raw, cv2.COLOR_RGB2GRAY)
    return {"mean_lum": float(g.mean()), "std_lum": float(g.std()),
            "p01_lum": float(np.percentile(g, 1)), "p99_lum": float(np.percentile(g, 99)),
            "frac_clipped_hi": float((g8 >= 250).mean()), "frac_clipped_lo": float((g8 <= 8).mean()),
            "lap_var": float(lap.var()),
            "mean_R": float(im[:, :, 0].mean()), "mean_G": float(im[:, :, 1].mean()),
            "mean_B": float(im[:, :, 2].mean()),
            "rb_ratio": float((im[:, :, 0].mean() + 1e-6) / (im[:, :, 2].mean() + 1e-6))}

POST_FEATS = ["mean_lum", "std_lum", "p01_lum", "p99_lum", "frac_clipped_hi", "frac_clipped_lo",
              "lap_var", "mean_R", "mean_G", "mean_B", "rb_ratio"]
print("recomputing capture statistics on the preprocessed images ...")
def stats_frame(standardise, which):
    return pd.DataFrame(Parallel(n_jobs=N_JOBS, verbose=1)(
        delayed(post_stats)(i, standardise, which) for i in range(len(master))))

post_raw = stats_frame(False, "neutral_fill")
post_std = stats_frame(True, "neutral_fill")
post_int = stats_frame(True, "interior")

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_predict
from sklearn.metrics import f1_score

PRE_FEATS = [c for c in ["min_side", "megapixels", "file_kb", "width", "height", "mean_lum",
                         "std_lum", "p01_lum", "p99_lum", "egg_mean_lum", "egg_std_lum", "mean_B",
                         "mean_G", "mean_R", "rb_ratio", "egg_area_frac", "egg_bbox_fill",
                         "frac_clipped_hi", "frac_clipped_lo", "egg_frac_clipped_hi",
                         "lap_var_global", "lap_var_egg"] if c in master.columns]

y4 = master["label4"].astype(str).to_numpy()
y2 = master["is_cracked"].to_numpy()
blocks = np.minimum((master["capture_order"].to_numpy() - 1) // 65, 9)

def probe(X, y, groups=None):
    clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.08,
                                         random_state=CFG.RANDOM_SEED)
    cv = (GroupKFold(n_splits=len(np.unique(groups))) if groups is not None
          else StratifiedKFold(N_FOLDS, shuffle=True, random_state=CFG.RANDOM_SEED))
    yp = cross_val_predict(clf, X, y, cv=cv, groups=groups)
    return round(float(f1_score(y, yp, average="macro")), 3)

STAGES = {
    "before preprocessing": master[PRE_FEATS].to_numpy(float),
    "after tight crop + resize": post_raw[POST_FEATS].to_numpy(float),
    "after crop + resize + standardisation": post_std[POST_FEATS].to_numpy(float),
    "interior crop + standardisation": post_int[POST_FEATS].to_numpy(float),
}
rows = []
for name, X in STAGES.items():
    rows.append({"pipeline stage": name,
                 "4-class (random split)": probe(X, y4),
                 "4-class (session-grouped)": probe(X, y4, blocks),
                 "binary (random split)": probe(X, y2),
                 "binary (session-grouped)": probe(X, y2, blocks)})
leak = pd.DataFrame(rows).set_index("pipeline stage")
leak.loc["majority baseline"] = [
    round(float(f1_score(y4, np.full_like(y4, "Intact"), average="macro")), 3), np.nan,
    round(float(f1_score(y2, np.zeros_like(y2), average="macro")), 3), np.nan]
savetable(leak, "t02_residual_leak_after_preprocessing",
          "Classification from capture statistics alone, before and after preprocessing")
S["residual_leak"] = leak.reset_index().to_dict("records")
print(leak.to_string())

before = leak.loc["before preprocessing", "4-class (random split)"]
after = leak.loc["after crop + resize + standardisation", "4-class (random split)"]
S["leak_before"], S["leak_after"] = float(before), float(after)
S["leak_reduction_pct"] = round(100 * (1 - after / before), 1)
print(f"\n>>> four-class metadata shortcut: {before} -> {after} "
      f"({S['leak_reduction_pct']:.0f}% reduction)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
names = list(STAGES)
x = np.arange(len(names)); w = 0.36
ax = axes[0]
ax.bar(x - w/2, [leak.loc[n, "4-class (random split)"] for n in names], w,
       label="random split", color="#C44E52")
ax.bar(x + w/2, [leak.loc[n, "4-class (session-grouped)"] for n in names], w,
       label="session-grouped", color="#937860")
ax.axhline(leak.loc["majority baseline", "4-class (random split)"], ls="--", color="#888", lw=1)
ax.set_xticks(x); ax.set_xticklabels([n.replace(" + ", "\n+ ") for n in names], fontsize=8)
ax.set_ylabel("macro F1 from capture statistics alone")
ax.set_title("(a) Four-class morphology"); ax.legend(fontsize=8); ax.set_ylim(0, 0.9)
for xi, n in zip(x - w/2, names):
    ax.text(xi, leak.loc[n, "4-class (random split)"] + 0.012,
            f"{leak.loc[n, '4-class (random split)']:.2f}", ha="center", fontsize=8)

ax = axes[1]
ax.bar(x - w/2, [leak.loc[n, "binary (random split)"] for n in names], w,
       label="random split", color="#4C72B0")
ax.bar(x + w/2, [leak.loc[n, "binary (session-grouped)"] for n in names], w,
       label="session-grouped", color="#64B5CD")
ax.axhline(leak.loc["majority baseline", "binary (random split)"], ls="--", color="#888", lw=1)
ax.set_xticks(x); ax.set_xticklabels([n.replace(" + ", "\n+ ") for n in names], fontsize=8)
ax.set_title("(b) Binary crack detection"); ax.legend(fontsize=8); ax.set_ylim(0, 0.9)
for xi, n in zip(x - w/2, names):
    ax.text(xi, leak.loc[n, "binary (random split)"] + 0.012,
            f"{leak.loc[n, 'binary (random split)']:.2f}", ha="center", fontsize=8)
fig.suptitle("Residual acquisition shortcut at each stage of the preprocessing pipeline",
             y=1.03, fontsize=12)
fig.tight_layout()
savefig(fig, "residual_leak", "Metadata-only classification before and after preprocessing")
plt.show()

---
## 4. Partitioning

Two split regimes are produced, and **both are reported in Chapter 5**. The gap between them is the
honest measure of how much the model depends on capture conditions.

**Primary — stratified with capture-block balancing.** Stratified jointly on `label4`, `age_class`
and capture block, so the test set mirrors the full dataset on all three. This is comparable to how
Cengel et al. (2025) and Cheng et al. (2024) report, and is the headline result.

**Secondary — leave-capture-block-out.** Whole capture sessions are held out, so the test session
was never seen during training. This is the pessimistic estimate and answers the question a farm
would actually ask: will it work on images I take tomorrow?

The test set is **sealed here and opened once, in notebook 07**. Nothing downstream may inspect it.

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

master["capture_block"] = np.minimum((master["capture_order"] - 1) // (len(master) // N_CAPTURE_BLOCKS),
                                     N_CAPTURE_BLOCKS - 1)
def build_strata(df, min_count=4):
    """Stratify as finely as the data allows, coarsening any stratum that is too small.

    Ideal is label x age x capture block. Where that leaves fewer than `min_count` eggs the
    stratum falls back to label x age, then to label alone, so no singleton stratum survives
    to break StratifiedShuffleSplit.
    """
    levels = [
        df["label4"].astype(str) + "|" + df["age_class"].astype(str) + "|" + df["capture_block"].astype(str),
        df["label4"].astype(str) + "|" + df["age_class"].astype(str),
        df["label4"].astype(str),
    ]
    out = levels[0].copy()
    for nxt in levels[1:]:
        too_small = out.map(out.value_counts()) < min_count
        if not too_small.any():
            break
        out = out.where(~too_small, nxt)
    # anything still rare is pooled into a single catch-all so the split cannot fail
    rare = out.map(out.value_counts()) < 2
    out = out.where(~rare, "__pooled__")
    return out

strat = build_strata(master)
print(f"stratification: {strat.nunique()} strata, smallest = {strat.value_counts().min()}")

sss = StratifiedShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=CFG.RANDOM_SEED)
dev_idx, test_idx = next(sss.split(np.zeros(len(master)), strat))
master["split"] = "dev"; master.loc[master.index[test_idx], "split"] = "test"

dev = master[master["split"].eq("dev")]
skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=CFG.RANDOM_SEED)
dev_strat = build_strata(dev, min_count=N_FOLDS)
master["fold"] = -1
for k, (_, va) in enumerate(skf.split(np.zeros(len(dev)), dev_strat)):
    master.loc[dev.index[va], "fold"] = k

# secondary regime: each capture block is a held-out group
master["group_fold"] = master["capture_block"]

print(f"test set: {int((master['split'] == 'test').sum())} eggs "
      f"({100*TEST_FRACTION:.0f}%), sealed until notebook 07")
print(f"development set: {int((master['split'] == 'dev').sum())} eggs in {N_FOLDS} folds\n")

chk = pd.concat([
    master.groupby("split", observed=True)["label4"].value_counts(normalize=True)
          .unstack().round(3).add_prefix("pct_"),
    master.groupby("split", observed=True)["age_class"].value_counts(normalize=True)
          .unstack().round(3).add_prefix("pct_"),
], axis=1)
savetable(chk, "t03_split_composition", "Class and age composition of the development and test sets")
print(chk.to_string())

fold_chk = master[master["fold"] >= 0].groupby("fold")["label4"].value_counts().unstack().fillna(0).astype(int)
savetable(fold_chk, "t04_fold_composition", "Class composition of each cross-validation fold")
print("\nfold composition:"); print(fold_chk.to_string())
S["test_n"] = int((master["split"] == "test").sum())
S["test_class_counts"] = master.loc[master["split"].eq("test"), "label4"].value_counts().to_dict()

In [ ]:
# --- sanity: is the test set distinguishable from the dev set? ------------
# If a classifier can tell test from dev using capture statistics, the split is not representative.
Xs = post_std[POST_FEATS].to_numpy(float)
is_test = (master["split"] == "test").astype(int).to_numpy()
sep = probe(Xs, is_test)
sep_base = round(float(f1_score(is_test, np.zeros_like(is_test), average="macro")), 3)
S["dev_test_separability"] = sep
S["dev_test_separability_baseline"] = sep_base
print(f"can capture statistics distinguish the test set from the development set?")
print(f"   macro F1 = {sep:.3f}   (majority baseline for this split ratio = {sep_base:.3f})")
if sep <= sep_base + 0.05:
    print("   -> the test set is representative; proceed.")
else:
    print("   -> the split selected an atypical subset. Re-draw with a different RANDOM_SEED")
    print("      and re-run this cell before continuing.")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
ax = axes[0]
ct = pd.crosstab(master["capture_block"], master["split"])
ct.plot(kind="bar", stacked=True, ax=ax, color=["#4C72B0", "#C44E52"], width=0.8, legend=True)
ax.set_xlabel("capture block"); ax.set_ylabel("eggs"); ax.set_title("(a) Test eggs across capture blocks")
ax.tick_params(axis="x", labelrotation=0)

ax = axes[1]
for sp, col in [("dev", "#4C72B0"), ("test", "#C44E52")]:
    d = master[master["split"].eq(sp)]["label4"].value_counts(normalize=True).reindex(CFG.LABEL4)
    ax.plot(CFG.LABEL4, d.values, "o-", color=col, label=sp, lw=2)
ax.set_ylabel("proportion"); ax.set_title("(b) Class balance matches"); ax.legend(fontsize=8)
ax.tick_params(axis="x", labelrotation=20)

ax = axes[2]
for sp, col in [("dev", "#4C72B0"), ("test", "#C44E52")]:
    ax.hist(post_std.loc[master["split"].eq(sp).to_numpy(), "lap_var"], bins=30, density=True,
            alpha=0.55, color=col, label=sp)
ax.set_xlabel("texture energy after standardisation"); ax.set_title(f"(c) Capture conditions match\nseparability F1 = {sep:.2f}")
ax.legend(fontsize=8)
fig.suptitle("Split validation", y=1.03, fontsize=12)
fig.tight_layout()
savefig(fig, "split_validation", "Validation that the held-out test set is representative")
plt.show()

---
## 5. Capture-matched sensitivity subset

Even after preprocessing, some residual shortcut remains. For Chapter 5 a sensitivity analysis is
prepared: a subset in which cracked and intact eggs are matched on their post-preprocessing capture
statistics, so that acquisition conditions carry as little class information as possible. Model
performance on this subset is the most conservative estimate available, and the difference from the
full test set is reported as a threat to validity.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier as HGB

ps = cross_val_predict(HGB(max_iter=300, random_state=CFG.RANDOM_SEED),
                       post_std[POST_FEATS].to_numpy(float), y2, cv=5,
                       method="predict_proba")[:, 1]
ps = np.clip(ps, 1e-4, 1 - 1e-4)
master["propensity"] = ps
master["ps_logit"] = np.log(ps / (1 - ps))

caliper = 0.2 * master["ps_logit"].std()
treated = master[master["is_cracked"].eq(1)].sort_values("ps_logit")
control = master[master["is_cracked"].eq(0)].copy()
used, pairs = set(), []
for _, row in treated.iterrows():
    cand = control[(~control["egg_id"].isin(used))
                   & ((control["ps_logit"] - row["ps_logit"]).abs() <= caliper)]
    if len(cand):
        m = cand.iloc[int((cand["ps_logit"] - row["ps_logit"]).abs().argmin())]
        used.add(m["egg_id"]); pairs.append((row["egg_id"], m["egg_id"]))

matched_ids = {a for a, _ in pairs} | {b for _, b in pairs}
master["matched_subset"] = master["egg_id"].isin(matched_ids)
mm = master[master["matched_subset"]]
resid = probe(post_std.loc[master["matched_subset"].to_numpy(), POST_FEATS].to_numpy(float),
              mm["is_cracked"].to_numpy())
print(f"matched pairs: {len(pairs)}  ->  {len(matched_ids)} eggs of {len(master)}")
print(f"class counts in matched subset: {mm['label4'].value_counts().to_dict()}")
print(f"residual capture-only probe on the matched subset: macro F1 = {resid:.3f}")
S["matched_n"] = int(len(matched_ids))
S["matched_class_counts"] = mm["label4"].value_counts().to_dict()
S["matched_residual_probe"] = resid
if resid > 0.45:
    print("\nNOTE: matching did not balance the groups well, which means the capture distributions")
    print("of cracked and intact eggs barely overlap. Report this subset as indicative only, and")
    print("rely primarily on the crack-masked control experiment in notebook 05.")

---
## 6. Export

In [ ]:
KEEP = ["egg_id", "path", "egg_status", "crack_pattern", "label4", "label4_idx", "is_cracked",
        "age_class", "age_idx", "capture_order", "capture_block", "split", "fold", "group_fold",
        "matched_subset", "propensity", "box_partial", "box_padded",
        "physiological_nutritional_pct", "mechanical_environmental_pct", "crack_region",
        "poly_n", "poly_area_pct", "poly_elongation_max", "poly_spread"]
KEEP = [c for c in KEEP if c in master.columns]
splits = master[KEEP].copy()
splits["cache_row"] = range(len(splits))
splits.to_csv(PROC / "splits.csv", index=False)
print(f"wrote {PROC / 'splits.csv'}  ({splits.shape})")

post_raw.assign(egg_id=master["egg_id"].values).to_csv(PROC / "post_stats_raw.csv", index=False)
post_std.assign(egg_id=master["egg_id"].values).to_csv(PROC / "post_stats_standardised.csv", index=False)

S["figures"], S["tables"] = FIGS, TABS
S["cache"] = {"path": str(CACHE_PATH), "side": CACHE_SIDE,
              "mb": round(CACHE_PATH.stat().st_size / 1e6, 1)}
def _safe(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.bool_): return bool(o)
    return str(o)
(OUT / "split_summary.json").write_text(json.dumps(S, indent=2, default=_safe))

L = ["=" * 70, "NOTEBOOK 02 DIGEST -- PREPROCESSING AND SPLITS", "=" * 70,
     f"generated: {S['generated_at']}", "",
     f"cache: {S['cache']['side']}px, {S['cache']['mb']} MB", ""]
L.append("RESIDUAL ACQUISITION SHORTCUT (macro F1 from capture statistics alone)")
for r in S["residual_leak"]:
    L.append(f"   {r['pipeline stage']:<42s} 4cls={r['4-class (random split)']}  "
             f"bin={r['binary (random split)']}  "
             f"4cls_grouped={r['4-class (session-grouped)']}")
L.append(f"   reduction: {S['leak_before']} -> {S['leak_after']}  ({S['leak_reduction_pct']}%)")
L.append("")
L.append(f"PARTIAL EGGS from crop geometry: see t01")
L.append(f"TEST SET: n={S['test_n']}  {S['test_class_counts']}")
L.append(f"dev/test separability from capture stats: {S['dev_test_separability']} "
         f"(baseline {S['dev_test_separability_baseline']})")
L.append(f"MATCHED SUBSET: n={S['matched_n']}  {S['matched_class_counts']}  "
         f"residual probe={S['matched_residual_probe']}")
L.append("=" * 70)
digest = "\n".join(L)
(OUT / "digest.txt").write_text(digest)
print(digest)